# 02 — Analysis Dataset
Notebook lanjutan dari `data_cleaning.ipynb`. Tujuannya menyiapkan **dataset analisis** untuk dashboard Excel:
kolom turunan dibuat di sini (bukan di Excel) supaya logika transformasi terdokumentasi dan bisa dijalankan ulang.

**Input:** `Clean_data/master_clean_orders_FIXED.xlsx` (20.404 order, 23 kolom)
**Output:** `Clean_data/analysis_dataset.xlsx` → dimuat ke Excel lewat Power Query sebagai `tbl_orders`

Aturan yang dipegang:
- Rate selalu memakai denominator `is_settled = 1` (250 order *Dalam Proses* Nov 2025 dikeluarkan).
- Tidak ada analisis Rupiah dan tidak ada analisis jam/hari (lihat batasan di README dashboard).

### Import Library

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

PATH_IN  = 'Clean_data/master_clean_orders_FIXED.xlsx'
PATH_OUT = 'Clean_data/analysis_dataset.xlsx'
MIN_N = 30   # batas minimum order settled agar rate suatu grup dianggap layak dibandingkan

## Load Data

In [2]:
df = pd.read_excel(PATH_IN)
print(df.shape)
assert df['order_id'].is_unique, 'order_id harus unik (1 baris = 1 order)'
assert not ((df['is_gagal_revenue'] == 1) & (df['is_settled'] == 0)).any(), \
    'Order gagal harus selalu settled, kalau tidak calculated field Rate di pivot salah'

(20404, 23)


## 1. Kolom Waktu untuk Slicer
`bulan_file` diambil dari nama file mentah, jadi aman untuk agregasi bulanan (berbeda dengan `waktu_pesanan` yang sebagian proxy).
Slicer Bulan cukup menampilkan nama bulan (`Jan`, `Feb`, …), tetapi teks seperti itu akan diurutkan alfabet oleh Excel (Agu, Apr, Des, …).
Solusinya, `bulan` disimpan sebagai **tanggal dummy** (tanggal 15, tahun 2000 — tahunnya tidak bermakna, hanya untuk urutan). Di Excel kolom ini diformat `[$-421]mmm`
sehingga tampil sebagai nama bulan Indonesia, sementara slicer mengurutkannya secara kronologis. Cara ini portabel: tidak bergantung pada custom list di komputer tertentu.

In [3]:
NAMA_BULAN = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']

df['tahun'] = df['bulan_file'].str[:4].astype(int)
no_bulan = df['bulan_file'].str[5:7].astype(int)
df['bulan'] = pd.to_datetime(pd.DataFrame({'year': 2000, 'month': no_bulan, 'day': 15}))

cek = df.groupby(['tahun', 'bulan']).size().unstack(0)
cek.index = [NAMA_BULAN[d.month - 1] for d in cek.index]   # tampilan seperti di slicer
cek

tahun,2024,2025
Jan,431.0,1150.0
Feb,389.0,957.0
Mar,453.0,628.0
Apr,492.0,824.0
Mei,620.0,819.0
Jun,697.0,615.0
Jul,1270.0,766.0
Agu,1293.0,942.0
Sep,1123.0,1131.0
Okt,1217.0,1205.0


## 2. Standarisasi Alasan Pembatalan
Ada pasangan label yang isinya sama tetapi teksnya berbeda (kemungkinan beda bahasa/versi aplikasi pembeli; kedua label muncul di 21–23 bulan, jadi bukan pergantian versi pada satu tanggal).
Hanya sinonim yang jelas yang digabung. `Lainnya` **tidak** digabung dengan `berubah pikiran` karena itu kategori sisa, bukan sinonim.

In [4]:
PETA_ALASAN = {
    'Dibatalkan oleh Pembeli. Alasan: Lainnya/ berubah pikiran': 'Berubah pikiran',
    'Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan yang Ada': 'Ubah pesanan',
    'Dibatalkan oleh Pembeli. Alasan: Perlu mengubah pesanan': 'Ubah pesanan',
    'Dibatalkan oleh Pembeli. Alasan: Need to change delivery address': 'Ubah alamat',
    'Dibatalkan oleh Pembeli. Alasan: Perlu mengubah alamat pengiriman': 'Ubah alamat',
    'Dibatalkan secara otomatis oleh sistem. Alasan: Pesanan belum dibayar': 'Belum dibayar',
    'Dibatalkan secara otomatis oleh sistem. Alasan: Pengiriman gagal': 'Pengiriman gagal',
    'Dibatalkan secara otomatis oleh sistem. Alasan: Penjual gagal mengirimkan pesanan tepat waktu': 'Penjual telat kirim',
    'Dibatalkan secara otomatis oleh sistem. Alasan: Penjual tidak mengatur pengiriman tepat waktu': 'Penjual telat atur kirim',
    'Dibatalkan oleh Pembeli. Alasan: Lainnya': 'Lainnya (pembeli)',
    'Dibatalkan oleh Pembeli. Alasan: Penjual tidak responsif terhadap pertanyaan Pembeli': 'Penjual tidak responsif',
    'Dibatalkan oleh Pembeli. Alasan: Perlu mengubah Voucher': 'Ubah voucher',
    'Dibatalkan oleh Pembeli. Alasan: Proses pembayaran sulit': 'Pembayaran sulit',
    'Dibatalkan oleh Pembeli. Alasan: Tidak ingin membeli lagi': 'Tidak ingin beli lagi',
    'Dibatalkan secara otomatis oleh sistem. Alasan: Paket hilang di perjalanan. Kompensasi yang memenuhi syarat telah dikreditkan ke Saldo Penjual-mu.': 'Paket hilang',
    'Dibatalkan oleh Pembeli. Alasan: Menemukan yang lebih murah': 'Ada yang lebih murah',
    'Dibatalkan oleh Penjual. Alasan: Produk habis': 'Produk habis',
    'Dibatalkan secara otomatis oleh sistem. Alasan: Lainnya': 'Lainnya (sistem)',
}
df['alasan_standar'] = df['alasan_pembatalan'].map(PETA_ALASAN)

batal = df['is_cancelled'] == 1
belum_terpetakan = df.loc[batal & df['alasan_standar'].isna(), 'alasan_pembatalan'].unique()
assert len(belum_terpetakan) == 0, f'Alasan baru belum dipetakan: {belum_terpetakan}'
assert df.loc[~batal, 'alasan_standar'].isna().all()

# tiap alasan standar harus berasal dari satu sumber pembatalan saja
assert (df[batal].groupby('alasan_standar')['cancel_source'].nunique() == 1).all()

df.loc[batal, 'alasan_standar'].value_counts().to_frame('n').assign(
    pct=lambda t: (t['n'] / t['n'].sum() * 100).round(2))

,n,pct
alasan_standar,,
Ubah pesanan,644,23.52
Berubah pikiran,639,23.34
Belum dibayar,431,15.74
Ubah alamat,379,13.84
Pengiriman gagal,247,9.02
Penjual telat kirim,113,4.13
Penjual telat atur kirim,63,2.30
Lainnya (pembeli),43,1.57
Penjual tidak responsif,33,1.21


## 3. Grup yang Layak Dibandingkan (n ≥ 30)
Kelayakan dihitung **secara global** (seluruh 23 bulan, order settled). Grup di bawah batas dikosongkan dan tidak tampil di chart.
- `metode_analisis`: metode pembayaran dengan n ≥ 30.
- `kategori_analisis`: hanya order dengan **1 kategori** (supaya rate bisa diatribusikan ke satu kategori), dan kategori itu n ≥ 30.

In [5]:
settled = df[df['is_settled'] == 1]

n_metode = settled['metode_pembayaran'].value_counts()
metode_layak = n_metode[n_metode >= MIN_N].index
df['metode_analisis'] = df['metode_pembayaran'].where(df['metode_pembayaran'].isin(metode_layak))

satu_kat = df['jumlah_kategori_berbeda'] == 1
n_kat = settled.loc[settled['jumlah_kategori_berbeda'] == 1, 'daftar_kategori'].value_counts()
kat_layak = n_kat[n_kat >= MIN_N].index
df['kategori_analisis'] = df['daftar_kategori'].where(satu_kat & df['daftar_kategori'].isin(kat_layak))

print(f'Metode layak: {len(metode_layak)} dari {len(n_metode)} '
      f'(dibuang: {n_metode[n_metode < MIN_N].to_dict()})')
print(f'Kategori layak: {len(kat_layak)} dari {len(n_kat)} kategori tunggal')
print(f"Order settled yang masuk analisis kategori: {df.loc[df['is_settled'] == 1, 'kategori_analisis'].notna().sum()}")

Metode layak: 8 dari 12 (dibuang: {'Pembayaran dibebaskan': 18, 'BCA OneKlik': 3, 'Cicilan Kartu Kredit': 3, 'Mitra Shopee': 1})
Kategori layak: 27 dari 37 kategori tunggal
Order settled yang masuk analisis kategori: 18582


## 4. Flag Artefak Opsi Pengiriman
Opsi generik tanpa kurir spesifik selalu 100% Batal karena kurir belum ditugaskan saat order dibatalkan — efek pencatatan, bukan temuan bisnis.
Flag ini **tidak** mengeluarkan order dari KPI (pembatalannya nyata); hanya dipakai saat menganalisis opsi pengiriman.

In [6]:
ARTEFAK_KIRIM = ['Hemat Kargo', 'Reguler (Cashless)', 'Instant (Versi Lama)', 'Kargo',
                 'Same Day', 'Next Day', 'Instant', 'GoSend Instant Prioritas']
df['is_artefak_kirim'] = df['opsi_pengiriman'].isin(ARTEFAK_KIRIM).astype(int)

art = df[df['is_artefak_kirim'] == 1]
print(f"Order artefak: {len(art)} | rate gagal: {art['is_gagal_revenue'].mean():.0%}")

Order artefak: 724 | rate gagal: 100%


## 5. Pilih Kolom & Validasi Akhir
Kolom yang sengaja **tidak** dibawa ke dashboard:
- `total_pembayaran_idr`: tidak konsisten pasca-retur, jadi dibuang agar tidak dipakai menghitung Rp loss.
- `waktu_pesanan`, `sumber_kolom_waktu`: sebagian proxy jadwal kirim, tidak aman untuk analisis jam/hari.
- `flag_anomali_bayar_nol`: bernilai 0 di semua baris.
- `kota`, `provinsi`, `opsi_pengiriman`, `alasan_pembatalan`: tidak dipakai di dashboard (alasan diganti `alasan_standar`).

In [7]:
KOLOM = ['order_id', 'bulan_file', 'tahun', 'bulan', 'status_efektif',
         'is_settled', 'is_gagal_revenue', 'is_cancelled', 'cancel_source', 'alasan_standar',
         'is_returned', 'return_type', 'metode_pembayaran', 'metode_analisis',
         'jumlah_kategori_berbeda', 'daftar_kategori', 'kategori_analisis', 'is_artefak_kirim']
out = df[KOLOM].sort_values(['bulan_file', 'order_id'], kind='stable').reset_index(drop=True)
assert (out['bulan'].dt.month == out['bulan_file'].str[5:7].astype(int)).all()

s = out[out['is_settled'] == 1]
cek = {
    'baris': (len(out), 20404),
    'order settled': (int(out['is_settled'].sum()), 20154),
    'order gagal-revenue': (int(out['is_gagal_revenue'].sum()), 2835),
    'order batal': (int(out['is_cancelled'].sum()), 2738),
    'rate overall (%)': (round(s['is_gagal_revenue'].mean() * 100, 2), 14.07),
    'rate Mei 2025 (%)': (round(s.loc[s['bulan_file'] == '2025-05', 'is_gagal_revenue'].mean() * 100, 2), 22.10),
}
for nama, (hasil, harapan) in cek.items():
    print(f"{'OK ' if hasil == harapan else 'BEDA'} {nama}: {hasil} (harapan {harapan})")
assert all(h == e for h, e in cek.values())

OK  baris: 20404 (harapan 20404)
OK  order settled: 20154 (harapan 20154)
OK  order gagal-revenue: 2835 (harapan 2835)
OK  order batal: 2738 (harapan 2738)
OK  rate overall (%): 14.07 (harapan 14.07)
OK  rate Mei 2025 (%): 22.1 (harapan 22.1)


In [8]:
out.to_excel(PATH_OUT, index=False, sheet_name='data')
print(f'Tersimpan: {PATH_OUT} -> {out.shape}')
out.head()

Tersimpan: Clean_data/analysis_dataset.xlsx -> (20404, 18)


,order_id,bulan_file,tahun,bulan,status_efektif,is_settled,is_gagal_revenue,is_cancelled,cancel_source,alasan_standar,is_returned,return_type,metode_pembayaran,metode_analisis,jumlah_kategori_berbeda,daftar_kategori,kategori_analisis,is_artefak_kirim
0,ORD_0006556,2024-01,2024,2000-01-15,Selesai,1,0,0,Tidak Batal,NaN,0,Tidak Retur,Saldo ShopeePay,Saldo ShopeePay,1,Aksesoris ID Card,Aksesoris ID Card,0
1,ORD_0006557,2024-01,2024,2000-01-15,Selesai,1,0,0,Tidak Batal,NaN,0,Tidak Retur,COD (Bayar di Tempat),COD (Bayar di Tempat),1,Celengan,Celengan,0
2,ORD_0006558,2024-01,2024,2000-01-15,Selesai,1,0,0,Tidak Batal,NaN,0,Tidak Retur,COD (Bayar di Tempat),COD (Bayar di Tempat),1,Plastik / Wadah Plastik,Plastik / Wadah Plastik,0
3,ORD_0006559,2024-01,2024,2000-01-15,Selesai,1,0,0,Tidak Batal,NaN,0,Tidak Retur,COD (Bayar di Tempat),COD (Bayar di Tempat),1,Rak / Rak Serbaguna,Rak / Rak Serbaguna,0
4,ORD_0006560,2024-01,2024,2000-01-15,Selesai,1,0,0,Tidak Batal,NaN,0,Tidak Retur,Online Payment,Online Payment,1,Celengan,Celengan,0
